# Merging Datasets

In this notebook, we will focus on merging the two datasets together into different dataframes in order to help with buidling our visualizations.

In [30]:
# import libraries
import pandas as pd

### Initialize helper code and read filtered data into data frames

In [31]:
%run helpers.ipynb
imdb_df, oscar_df = read_data('imdb_filtered.csv', 'oscar_filtered.csv')

## Joining Oscar data with IMDb data

We want to begin joining the Oscar dataset with the IMDb dataset, using **FilmId** and **id** as keys. These columns both contain the IMDb Id of each film.

It is important to note that this will be done using a left join, as we want to add additional information from the IMDb dataset to the list of Oscar nominees in order to show more details about each film.

The **id** column from the IMDb dataset will be dropped as we don't want duplicate columns.

In [32]:
# join datasets together by film id
# imdb is id; oscar is FilmId
merged_df = pd.merge(oscar_df, imdb_df, left_on='FilmId', right_on='id', how='left', indicator=True)
merged_df = merged_df.drop(columns=['id'])
merged_df.head(10)


,Unnamed: 0_x,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,...,budget,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages,_merge
0,0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,...,NaN,NaN,NaN,NaN,1928-01-29,['United States'],['First National Pictures'],['Drama'],"['None', 'English']",both
1,1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,...,NaN,NaN,NaN,NaN,1927-09-01,['United States'],['First National Pictures'],"['Boxing', 'Drama', 'Romance', 'Sport', 'War']","['None', 'English']",both
2,2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command,tt0019071,Emil Jannings,Emil Jannings,...,NaN,NaN,NaN,NaN,1928-01-21,['United States'],['Paramount Pictures'],"['Political Drama', 'Showbiz Drama', 'Tragedy'...","['None', 'English']",both
3,3,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,...,NaN,NaN,NaN,859900.0,1927-10-01,['United States'],['Paramount Pictures'],['Drama'],"['None', 'English']",both
4,4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,...,NaN,NaN,NaN,NaN,1928-06-04,['United States'],['DeMille Pictures Corporation'],['Drama'],['None'],both
5,5,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven,tt0018379,Janet Gaynor,Janet Gaynor,...,1300000.0,NaN,NaN,NaN,1927-10-30,['United States'],"['Frank Borzage Production', 'Fox Film Corpora...","['Drama', 'Romance']","['None', 'English']",both
6,6,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Street Angel,tt0019429,Janet Gaynor,Janet Gaynor,...,NaN,NaN,NaN,3706000.0,1928-08-19,['United States'],['Fox Film Corporation'],['Drama'],"['None', 'English']",both
7,7,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sunrise,tt0018455,Janet Gaynor,Janet Gaynor,...,200000.0,NaN,121848.0,NaN,1927-11-04,['United States'],['Fox Film Corporation'],"['Dark Romance', 'Psychological Drama', 'Drama...","['None', 'English']",both
8,8,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,...,650000.0,NaN,NaN,NaN,1928-01-07,['United States'],['Gloria Swanson Pictures'],['Drama'],['English'],both
9,9,1,1927/28,Production,ART DIRECTION,ART DIRECTION,Sunrise,tt0018455,Rochus Gliese,Rochus Gliese,...,200000.0,NaN,121848.0,NaN,1927-11-04,['United States'],['Fox Film Corporation'],"['Dark Romance', 'Psychological Drama', 'Drama...","['None', 'English']",both


### Join Failures
There are now 27 columns in this merged data frame. Ten columns are from the Oscar dataset, and 17 are from the IMDb dataset. However, we want to know how much data was not joined. There may be films in the list of Oscar nominees that are missing from the IMDb dataset.

In [33]:
# here we want to see a list of join failures, i.e. Oscar nominees missing from the imdb dataset
not_joined = (
    merged_df
    .loc[merged_df['_merge'] == 'left_only']
    .groupby('Category')
    .size()
    .sort_values(ascending=False)
)

not_joined

Category
FOREIGN LANGUAGE FILM                                  21
MUSIC (Original Song)                                   8
CINEMATOGRAPHY                                          5
MUSIC (Song--Original for the Picture)                  3
SPECIAL AWARD                                           3
MUSIC (Song)                                            3
MUSIC (Scoring of a Musical Picture)                    3
ANIMATED FEATURE FILM                                   3
COSTUME DESIGN                                          3
INTERNATIONAL FEATURE FILM                              2
DIRECTING                                               2
WRITING (Adapted Screenplay)                            2
ART DIRECTION                                           2
ACTRESS IN A SUPPORTING ROLE                            2
PRODUCTION DESIGN                                       1
WRITING                                                 1
SOUND RECORDING                                         1
SOUND

In [34]:
# total join failures
print(not_joined.sum())

79


We can see that of the 9,000+ rows in the Oscar dataset, 79 of them failed to appear in the IMDb dataset. The highest category was for Foreign Language Film. Overall, we can consider this number of missing rows in the merged dataset to be fine since it is relatively low. Additionally, most of the categories listed are missing only 1-5 films, which should not skew our dataset.

One possible reason for these join failures could be related to the IMDb dataset containing a list of the Top 500-600 films from the years 1920-2025. Therefore, it could be that these 79 films were not voted highly on IMDb to have been included in the raw dataset.



## Dropping Oscar nominees from IMDb data

Another dataframe that will be useful in our investigation is the list of films in our IMDb dataset that were **not** nominated for any Oscar awards. This is particularly important for helping us investigate audience popularity against award/prestige bias.

In [35]:
# create a dataframe with movies in imdb dataset that were not nominated for oscars
non_oscar_noms = imdb_df[~imdb_df['id'].isin(oscar_df['FilmId'])]
non_oscar_noms.head()

,Unnamed: 0,id,title,rating,votes,meta_score,writers,directors,stars,budget,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages
0,0,tt0011370,Klostret i Sendomir,6.9,485,NaN,"['Franz Grillparzer', 'Victor Sjöström']",['Victor Sjöström'],"['Tore Svennberg', 'Tora Teje', 'Richard Lund'...",NaN,NaN,NaN,NaN,1920-01-01,['Sweden'],['Svenska Biografteatern AB'],['Drama'],['None']
1,1,tt0011413,The Lost City,4.7,35,NaN,['Frederick Chapin'],['E.A. Martin'],"['Juanita Hansen', 'George Chesebro', 'Frank C...",NaN,NaN,NaN,NaN,1920-01-01,['United States'],['Selig Polyscope Company'],"['Action', 'Adventure']","['None', 'English']"
2,2,tt0276209,Hypnose,7.0,19,NaN,['Karl Schneider'],['Richard Eichberg'],"['Lee Parry', 'Gertrud de Lalsky', 'Karl Halde...",NaN,NaN,NaN,NaN,1920-01-03,['Germany'],['Richard Eichberg-Film GmbH'],"['Drama', 'Mystery']",['None']
3,3,tt0010495,My Husband's Other Wife,5.3,17,NaN,['Stanley Olmstead'],['J. Stuart Blackton'],"['Sylvia Breamer', 'Robert Gordon', 'Warren Ch...",NaN,NaN,NaN,NaN,1920-01-04,['United States'],['J. Stuart Blackton Feature Pictures'],"['Drama', 'Romance']",['None']
4,4,tt0010502,Nachtgestalten,5.6,25,NaN,"['Richard Oswald', 'Karl Hans Strobl']",['Richard Oswald'],"['Paul Wegener', 'Reinhold Schünzel', 'Erna Mo...",NaN,NaN,NaN,NaN,1920-01-09,['Germany'],['Richard-Oswald-Produktion'],['Horror'],"['None', 'German']"


We have managed to filter out around 5,000 films from our IMDb dataset. Note, this number won't be the shape of the Oscar dataset (9,073 rows) because the same film can be nominated multiple times for an Oscar.

In [36]:
non_oscar_noms.shape

(55684, 18)

In [38]:
write_joined(merged_df, non_oscar_noms)